In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
from lorentz import Lorentz

# load ENAO checkpoint and vocabulary
vocab = pickle.load(open("enoa_vocab.pkl", "rb"))
genre_to_idx = {genre: idx for idx, genre in enumerate(vocab)}
checkpoint = torch.load("ckpt/final_enoa_graph.ckpt", map_location="cpu")
embeddings = checkpoint["table.weight"].numpy()
print(f"ENAO vocabulary size: {len(vocab)} genres")

# load collaboration dataset
df = pd.read_csv("code/dataset/Original/global/global-genre_network-2018.csv", sep="\t")

# remove self loops and extract unique genres
df = df[df["source"] != df["target"]].copy()
collab_genres = set(df["source"].unique()) | set(df["target"].unique())
print(f"Unique genres in collaboration dataset: {len(collab_genres)}")

# extract and save embeddings for matching genres
matched = {}
unmatched = []
for genre in collab_genres:
    if genre in genre_to_idx:
        idx = genre_to_idx[genre] + 1
        matched[genre] = embeddings[idx]
    else:
        unmatched.append(genre)

print(f"Matched: {len(matched)} genres")
print(f"Unmatched (will be skipped): {len(unmatched)} genres")
if unmatched:
    print("Skipped genres:")
    for g in sorted(unmatched):
        print(f"  - {g}")

# save extracted embeddings for future use
pickle.dump(matched, open("collab_genre_embeddings.pkl", "wb"))
print("Saved extracted embeddings to collab_genre_embeddings.pkl")

# define Lorentz distance
def lorentz_scalar_product(u, v):
    return -u[0]*v[0] + np.dot(u[1:], v[1:])

def lorentz_distance(u, v):
    inner = lorentz_scalar_product(u, v)
    inner = min(inner, -1.0)
    return float(np.arccosh(-inner))

# compute distances for each collaboration pair
distances = []
skipped_rows = []
for _, row in df.iterrows():
    source = row["source"]
    target = row["target"]
    if source not in matched:
        skipped_rows.append((source, target, f"'{source}' not in ENAO embeddings"))
        distances.append(None)
        continue
    if target not in matched:
        skipped_rows.append((source, target, f"'{target}' not in ENAO embeddings"))
        distances.append(None)
        continue
    d = lorentz_distance(matched[source], matched[target])
    distances.append(d)

df["hyperbolic_distance"] = distances

# save results
df_clean = df.dropna(subset=["hyperbolic_distance"]).copy()
print(f"\nTotal collaboration pairs: {len(df)}")
print(f"Pairs skipped: {len(skipped_rows)}")
print(f"Pairs with distance computed: {len(df_clean)}")

if skipped_rows:
    print("\nSkipped rows log:")
    for source, target, reason in skipped_rows:
        print(f"  - {source} → {target}: {reason}")

df_clean.to_csv("collaborations_with_distances2018.csv", index=False)
print("\nSaved to collaborations_with_distances2018.csv")
print(df_clean[["source", "target", "weight", "avg_streams", "hyperbolic_distance"]].head(10))

collab_graph enoa_graph
ENAO vocabulary size: 6280 genres
Unique genres in collaboration dataset: 295
Matched: 284 genres
Unmatched (will be skipped): 11 genres
Skipped genres:
  - afro dancehall
  - baile pop
  - deep pop r&b
  - francoton
  - indie cafe pop
  - kwaito house
  - latin
  - lgbtq+ hip hop
  - nc hip hop
  - ninja
  - vapor trap
Saved extracted embeddings to collab_genre_embeddings.pkl

Total collaboration pairs: 2792
Pairs skipped: 211
Pairs with distance computed: 2581

Skipped rows log:
  - latin → reggaeton: 'latin' not in ENAO embeddings
  - latin → tropical: 'latin' not in ENAO embeddings
  - latin → trap latino: 'latin' not in ENAO embeddings
  - latin → reggaeton flow: 'latin' not in ENAO embeddings
  - latin → latin hip hop: 'latin' not in ENAO embeddings
  - latin → latin pop: 'latin' not in ENAO embeddings
  - latin → pop: 'latin' not in ENAO embeddings
  - latin → puerto rican pop: 'latin' not in ENAO embeddings
  - dance pop → latin: 'latin' not in ENAO embe